In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
# this thread provides a id for the user_chat if new chat open then new id will be given
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'A classic play on words. This joke is funny because it uses a pun to create a clever connection between the setup and the punchline. \n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty" subverts this expectation by using the word "crusty" in a double meaning. \n\nIn one sense, "crusty" can describe someone who is irritable or gruff, which fits with the idea of being in a bad mood. However, "crusty" is also a literal descriptor of a pizza\'s crust, which is the outer, often crunchy layer of the pizza. \n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the pizza\'s emotional state and its physical characteristics. The joke relies on this wordplay to create a lighthearted a

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is funny because it uses a pun to create a clever connection between the setup and the punchline. \n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty" subverts this expectation by using the word "crusty" in a double meaning. \n\nIn one sense, "crusty" can describe someone who is irritable or gruff, which fits with the idea of being in a bad mood. However, "crusty" is also a literal descriptor of a pizza\'s crust, which is the outer, often crunchy layer of the pizza. \n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the pizza\'s emotional state and its physical characteristics. The joke relies on this wordplay to crea

## want to see the history of the states

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is funny because it uses a pun to create a clever connection between the setup and the punchline. \n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty" subverts this expectation by using the word "crusty" in a double meaning. \n\nIn one sense, "crusty" can describe someone who is irritable or gruff, which fits with the idea of being in a bad mood. However, "crusty" is also a literal descriptor of a pizza\'s crust, which is the outer, often crunchy layer of the pizza. \n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the pizza\'s emotional state and its physical characteristics. The joke relies on this wordplay to cre

## new thread ,means new chat

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.',
 'explanation': 'A clever play on words. This joke is funny because it uses a common phrase associated with romantic relationships, "tangled up," and gives it a literal twist. In this case, the spaghetti is afraid of getting "tangled up" because, as a long, thin, and flexible food item, it can easily become knotted or entwined with other things. \n\nThe humor comes from the double meaning of the phrase. In relationships, "tangled up" typically means to be deeply involved or complicated, often in a way that\'s difficult to escape. But for spaghetti, "tangled up" is a literal concern, as it can easily become knotted or twisted. The joke relies on this wordplay to create a humorous connection between the setup (the spaghetti refusing to get married) and the punchline (the fear of getting tangled up). The result is a lighthearted and amusing joke tha

In [13]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is funny because it uses a pun to create a clever connection between the setup and the punchline. \n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty" subverts this expectation by using the word "crusty" in a double meaning. \n\nIn one sense, "crusty" can describe someone who is irritable or gruff, which fits with the idea of being in a bad mood. However, "crusty" is also a literal descriptor of a pizza\'s crust, which is the outer, often crunchy layer of the pizza. \n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the pizza\'s emotional state and its physical characteristics. The joke relies on this wordplay to crea

In [ ]:
# here you can see no change in the history og the thread_1
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is funny because it uses a pun to create a clever connection between the setup and the punchline. \n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a little crusty" subverts this expectation by using the word "crusty" in a double meaning. \n\nIn one sense, "crusty" can describe someone who is irritable or gruff, which fits with the idea of being in a bad mood. However, "crusty" is also a literal descriptor of a pizza\'s crust, which is the outer, often crunchy layer of the pizza. \n\nThe humor comes from the unexpected twist on the word\'s meaning, creating a clever and silly connection between the pizza\'s emotional state and its physical characteristics. The joke relies on this wordplay to cre

# Time travel
you have to provide the id first then invoke with none state

In [17]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f0ed49b-317b-6b94-8001-7acad3754a27"}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0ed49b-317b-6b94-8001-7acad3754a27'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-01-09T10:55:24.155790+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0ed49b-2c85-6808-8000-fd59a4d9513f'}}, tasks=(PregelTask(id='2b1118da-a980-a710-588c-57cea9a9d98b', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result={'explanation': 'A classic play on words. This joke is funny because it uses a pun to create a clever connection between the setup and the punchline. \n\nThe setup "Why was the pizza in a bad mood?" primes the listener to expect a reason related to the pizza\'s emotional state. The punchline "Because it was feeling a lit

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f0ed49b-317b-6b94-8001-7acad3754a27"}})

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'A classic play on words. This joke is an example of a pun, which is a type of wordplay that uses multiple meanings of a word to create humor.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. In one sense, "crusty" can describe someone or something that is irritable, grumpy, or in a bad mood. However, in the context of a pizza, "crusty" also refers to the crunchy, outer layer of the pizza, known as the crust.\n\nThe joke relies on this dual meaning to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The wordplay on "crusty" creates a clever and amusing link between the pizza\'s emotional state and its physical characteristics, making for a lighthearted and amusing joke.'}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is an example of a pun, which is a type of wordplay that uses multiple meanings of a word to create humor.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. In one sense, "crusty" can describe someone or something that is irritable, grumpy, or in a bad mood. However, in the context of a pizza, "crusty" also refers to the crunchy, outer layer of the pizza, known as the crust.\n\nThe joke relies on this dual meaning to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The wordplay on "crusty" creates a clever and amusing link between the pizza\'s emotional state and its physical characteristics, making for a lighthearted and amusing joke.'}, next=(), config={'configurable': {'thread_id': '1',

# update the state-
give the chackpoint and with new topic

In [20]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f0ed49b-2c7e-62db-bfff-a5850897f401", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0ed4ad-e912-696b-8000-950b04556b8c'}}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0ed4ad-e912-696b-8000-950b04556b8c'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-01-09T11:03:46.590346+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0ed49b-2c7e-62db-bfff-a5850897f401'}}, tasks=(PregelTask(id='816345d1-4103-3071-e131-b5e007a4eb04', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is an example of a pun, which is a type of wordplay that uses multiple meanings of a word to create humor.\n\nIn this joke, the phrase "feeling a little crusty" has a double meaning. In one sense, "cru

here provide the new checkpoint id from where you want to start-
 which is updated when you update the state

In [22]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f0ed4ad-e912-696b-8000-950b04556b8c"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little "crunchy" on the outside and "empty" on the inside.',
 'explanation': 'A clever joke. This joke is a play on words, using the physical characteristics of a samosa (a type of fried or baked pastry) to make a humorous comment on emotional well-being.\n\nThe punchline "it was feeling a little \'crunchy\' on the outside and \'empty\' on the inside" is a clever double entendre. In a literal sense, a samosa is typically crunchy on the outside (due to its fried or baked exterior) and empty on the inside (as it is often filled with a hollow space or a small amount of filling).\n\nHowever, the joke also uses these physical characteristics to make a metaphorical comment on emotional state. "Feeling crunchy on the outside" can be interpreted as feeling tough, hardened, or put-together on the surface, while "feeling empty on the inside" suggests a sense of emotional hollowness, emptiness, or lack of 

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little "crunchy" on the outside and "empty" on the inside.', 'explanation': 'A clever joke. This joke is a play on words, using the physical characteristics of a samosa (a type of fried or baked pastry) to make a humorous comment on emotional well-being.\n\nThe punchline "it was feeling a little \'crunchy\' on the outside and \'empty\' on the inside" is a clever double entendre. In a literal sense, a samosa is typically crunchy on the outside (due to its fried or baked exterior) and empty on the inside (as it is often filled with a hollow space or a small amount of filling).\n\nHowever, the joke also uses these physical characteristics to make a metaphorical comment on emotional state. "Feeling crunchy on the outside" can be interpreted as feeling tough, hardened, or put-together on the surface, while "feeling empty on the inside" suggests a sense of emotional hollowness, em